# Chapter 4 Notebook 1 
### The Garces model is not truly bilevel

This notebook implements the _linearized_ model from [1, Sec. II.C].

The mathematical bi-level model has been been reduced to a single-level model
by constructing the dual of the lower level and using the strong duality theorem
on the lower level. The strong duality theorem can be used here because when
we treat the transmission variables as given parameters for the lower level, the
problem is linear; that is to say, the strong duality theorem gives us a
necessary and sufficient condition for optimality in the lower level, given
specific upper-level decisions. Considered as a whole, however, the problem is
instead non-linear as the constraints of the lower level include terms which
multiply upper-level decision variables with lower-level decision variables
(e.g. constraint (7) for line flow).

In the model in this notebook, these non-linearities are removed. Constraints
(7), (8), and (9) are replaced by constraints (40) and (41), which use a big-M
constraint to linearize the line flow limit constraints. Dual constraints (26) 
and (27) are also linearized by new constraints (42), (43), (44), and (45).

This version is fully modularized so that the model is built
separately from results etc.

An additional aim is to see if the single-level model gives the same results.
This indeed appears to be the case.

## References
[1] Garces, L.P et al. “A Bilevel Approach to Transmission Expansion Planning Within a Market Environment.” *IEEE Transactions on Power Systems* 24.3 (2009): 1513–1522.

In [ ]:
# IMPORTS
# --------

# Standard libraries
import importlib
import pandas as pd
import numpy as np

import networkx as nx

# Optimizer
import xpress as xp
# xp.init('C:/Program Files/xpressmp/bin/xpauth_personal.xpr') # for local version
xp.init('C:/Program Files/xpressmp/bin/xpauth_uoe2.xpr')

# Data
import garces_data_module as data

# Model
import garces_model as model

importlib.reload(data)
importlib.reload(model)

<module 'garces_et_al_model_7d' from 'c:\\Users\\ajmac\\Documents\\ORMSc\\Code\\Required\\garces_et_al_model_7d.py'>

Here the model results are replicated using the bilevel approach.

In [2]:
data_parameters = {"Gamma_max": 1000}

data.load_data(parameters = data_parameters)
# data.load_data(scenario_list=[1])

# PICK ONE SCENARIO, comment out the others:
new_lines = []
# new_lines = [9,21,24,26] # min cost scenario 1 GARCES
# new_lines = [9,24,26,41] # min cost scenario 1 according to AJM (garces_et_al_02d.ipynb)
# new_lines = [14,21,26,29,44] # min cost scenario 2 GARCES
# new_lines = [9,14,26,29] # min cost scenario 3 GARCES

# "bilevel = True" and "alternative_objective = False" is the vanilla Garces:
bilevel = True
alternative_objective = False

model_params = {"new_lines": new_lines, 
                "bilevel"  : bilevel,
                "alternative_objective" : alternative_objective
                }

solver_options = {"outputlog": 0}

sol_dict = model.run_model(data.DataStore, model_params=model_params, solver_options=solver_options)

In [3]:
print("Variables/Expressions: ")
print(", ".join([k for k in sol_dict.keys()]))

Variables/Expressions: 
x, d, g, f, r, theta, lambda_s, phi, phi_max, phi_min, varphi_max, beta_max, alpha_max, rho, xi_max, xi_min, chi, phi_minus, sw_ls, tics, lambda_s_bar, lambda_bar


In [4]:
lines_now_open = sol_dict['x']
soln_line_list = [k for k,v in lines_now_open.items() if v >=0.99]

new_lines = [k for k in soln_line_list if k not in data.existing_lines]

# Print the new lines we build:
print("NEW LINES:")
print("----------")
for k in new_lines: # If the line is open...
    print(f"Line {k}: ({data.o[k]}-{data.r[k]})") # ... list it here!

# Print the total investment cost:
print("")
print(f"Total Investment Cost : EUR {sol_dict['tics']/10**6:.2f} million")

# If the plan has been based on a single scenario, print the social welfare in
# that specific scenario:
if len(data.scenarios) == 1:
    print("")
    print(f"Social Welfare: EUR {data.sigma*sum(sol_dict['sw_ls'].values())/10**6:.2f} million")

# Get the total load shed:
print("")
print(f"Total load shed: {sum(sol_dict['r'].values())} MW")

# Calculate the average social welfare ACROSS SCENARIOS for those fixed lines:
data.load_data(parameters=data_parameters)

# Reset the model parameters. In particular make sure that we force the lines
# to be open that were determined in the plan above.
model_params = {"new_lines": new_lines, 
                "bilevel"  : bilevel,
                "alternative_objective" : alternative_objective
                }

solver_options = {"outputlog": 0}

sol_dict = model.run_model(data.DataStore, model_params=model_params, solver_options=solver_options)


sw = {k: data.sigma*v/10**6 for k,v in sol_dict['sw_ls'].items()}
print("")
for w in data.scenarios:
    print(f"Social Welfare in Scenario {w}: {sw[w]:.2f}")

SW = {k: data.sigma*data.delta[k]*v for k,v in sol_dict['sw_ls'].items()}

print("")
print(f"Average Social Welfare: EUR {sum(SW.values())/10**6:.2f} million")
print("")
print(f"    *N.B. Scenarios weights are: {list(data.delta.values())}*")

NEW LINES:
----------
Line 14: (4-6)
Line 21: (2-3)
Line 29: (4-6)
Line 41: (3-5)
Line 44: (4-6)

Total Investment Cost : EUR 25.10 million

Total load shed: 0.0 MW

Social Welfare in Scenario 1: 278.49
Social Welfare in Scenario 2: 307.79
Social Welfare in Scenario 3: 306.09

Average Social Welfare: EUR 292.71 million

    *N.B. Scenarios weights are: [0.5, 0.25, 0.25]*


We now do the same without employing the bilevel approach.
The same results are observed.

In [5]:
data_parameters = {"Gamma_max": 1000}

data.load_data(parameters = data_parameters)
# data.load_data(scenario_list=[1])

# PICK ONE SCENARIO, comment out the others:
new_lines = []
# new_lines = [9,21,24,26] # min cost scenario 1 GARCES
# new_lines = [9,24,26,41] # min cost scenario 1 according to AJM (garces_et_al_02d.ipynb)
# new_lines = [14,21,26,29,44] # min cost scenario 2 GARCES
# new_lines = [9,14,26,29] # min cost scenario 3 GARCES

# "bilevel = True" and "alternative_objective = False" is the vanilla Garces:
bilevel = False
alternative_objective = False

model_params = {"new_lines": new_lines, 
                "bilevel"  : bilevel,
                "alternative_objective" : alternative_objective
                }

solver_options = {"outputlog": 0}

sol_dict = model.run_model(data.DataStore, model_params=model_params, solver_options=solver_options)

In [6]:
print("Variables/Expressions: ")
print(", ".join([k for k in sol_dict.keys()]))

Variables/Expressions: 
x, d, g, f, r, theta, sw_ls, tics


In [7]:
lines_now_open = sol_dict['x']
soln_line_list = [k for k,v in lines_now_open.items() if v >=0.99]

new_lines = [k for k in soln_line_list if k not in data.existing_lines]

# Print the new lines we build:
print("NEW LINES:")
print("----------")
for k in new_lines: # If the line is open...
    print(f"Line {k}: ({data.o[k]}-{data.r[k]})") # ... list it here!

# Print the total investment cost:
print("")
print(f"Total Investment Cost : EUR {sol_dict['tics']/10**6:.2f} million")

# If the plan has been based on a single scenario, print the social welfare in
# that specific scenario:
if len(data.scenarios) == 1:
    print("")
    print(f"Social Welfare: EUR {data.sigma*sum(sol_dict['sw_ls'].values())/10**6:.2f} million")

# Get the total load shed:
print("")
print(f"Total load shed: {sum(sol_dict['r'].values())} MW")

# Calculate the average social welfare ACROSS SCENARIOS for those fixed lines:
data.load_data(parameters=data_parameters)

# Reset the model parameters. In particular make sure that we force the lines
# to be open that were determined in the plan above.
model_params = {"new_lines": new_lines, 
                "bilevel"  : bilevel,
                "alternative_objective" : alternative_objective
                }

solver_options = {"outputlog": 0}

sol_dict = model.run_model(data.DataStore, model_params=model_params, solver_options=solver_options)


sw = {k: data.sigma*v/10**6 for k,v in sol_dict['sw_ls'].items()}
print("")
for w in data.scenarios:
    print(f"Social Welfare in Scenario {w}: {sw[w]:.2f}")

SW = {k: data.sigma*data.delta[k]*v for k,v in sol_dict['sw_ls'].items()}

print("")
print(f"Average Social Welfare: EUR {sum(SW.values())/10**6:.2f} million")
print("")
print(f"    *N.B. Scenarios weights are: {list(data.delta.values())}*")

NEW LINES:
----------
Line 14: (4-6)
Line 26: (3-5)
Line 29: (4-6)
Line 36: (2-3)
Line 44: (4-6)

Total Investment Cost : EUR 25.10 million

Total load shed: 0.0 MW

Social Welfare in Scenario 1: 278.49
Social Welfare in Scenario 2: 307.79
Social Welfare in Scenario 3: 306.09

Average Social Welfare: EUR 292.71 million

    *N.B. Scenarios weights are: [0.5, 0.25, 0.25]*
